In [1]:
import json
from pathlib import Path
from safetensors.torch import load_file

load_dir = Path("/workspace/results")

with open(load_dir / "patient_graphs_meta.json") as f:
    meta = json.load(f)
loaded_sample_ids = meta["sample_ids"]
loaded_ordered_regions = meta["ordered_regions"]

loaded_stacked = load_file(load_dir / "patient_graphs.safetensors")["patient_graphs"]

# rebuild the per-patient dict keyed the same way as the in-memory patient_graphs
loaded_patient_graphs = {s: loaded_stacked[i] for i, s in enumerate(loaded_sample_ids)}

print(f"Loaded {loaded_stacked.shape} for {len(loaded_sample_ids)} samples, {len(loaded_ordered_regions)} regions")


Loaded torch.Size([536, 90399, 2]) for 536 samples, 90399 regions


In [2]:
ordered_regions = meta["ordered_regions"]
print(f"Total regions loaded: {len(ordered_regions)}")

Total regions loaded: 90399


In [3]:
base_genes = set()
for feature in ordered_regions:
    gene = feature.split('_')[0] 
    base_genes.add(gene)
unique_genes = sorted(list(base_genes))
print(f"Unique base genes to query against STRING: {len(unique_genes)}")

Unique base genes to query against STRING: 19577


In [4]:
import cudf
import torch
import itertools
import json
from torch_geometric.utils import add_self_loops

print("Loading patient metadata...")
with open("/workspace/results/patient_graphs_meta.json", "r") as f:
    ordered_regions = json.load(f)["ordered_regions"]

# Build base gene mapping (Gene -> list of node indices)
gene_to_indices = {}
for idx, feature in enumerate(ordered_regions):
    gene = feature.split('_')[0]
    if gene not in gene_to_indices:
        gene_to_indices[gene] = []
    gene_to_indices[gene].append(idx)

unique_genes = list(gene_to_indices.keys())

# ==========================================
# 1. Process the STRING Database locally
# ==========================================
print("Loading STRING links and filtering by high confidence...")
# Load the raw connections and filter immediately to save memory
links_df = cudf.read_csv("/workspace/data/9606.protein.links.v12.0.txt.gz", sep=" ")
links_df = links_df[links_df['combined_score'] >= 700]

print("Loading STRING aliases and matching to our genes...")
# Aliases file is tab-separated and contains the mapping from ENSP to standard Gene Symbols
aliases_df = cudf.read_csv("/workspace/data/9606.protein.aliases.v12.0.txt.gz", sep="\t", header=0)

# We only care about the aliases that match the genes in our dataset
valid_aliases = aliases_df[aliases_df['alias'].isin(unique_genes)]

# Deduplicate to prevent mapping explosions (take the first ENSP ID for a gene)
mapping_df = valid_aliases.drop_duplicates(subset=['#string_protein_id'])[['#string_protein_id', 'alias']]
mapping_df.columns = ['protein', 'gene']

print("Merging interactions...")
# Map protein1 to gene1
merged_df = links_df.merge(mapping_df, left_on='protein1', right_on='protein', how='inner')
merged_df = merged_df.rename(columns={'gene': 'gene1'})

# Map protein2 to gene2
merged_df = merged_df.merge(mapping_df, left_on='protein2', right_on='protein', how='inner')
merged_df = merged_df.rename(columns={'gene': 'gene2'})

# Bring the final gene-to-gene edges back to the CPU for the index building
edges_cpu = merged_df[['gene1', 'gene2']].to_pandas()

# ==========================================
# 2. Build the PyTorch edge_index
# ==========================================
source_nodes = []
target_nodes = []

print("Translating global edges to node indices...")
# Add inter-gene interactions (STRING data)
for _, row in edges_cpu.iterrows():
    gene_a, gene_b = row['gene1'], row['gene2']
    # Skip self-references in STRING (we handle intra-gene loops separately)
    if gene_a == gene_b: 
        continue
        
    for idx_a in gene_to_indices[gene_a]:
        for idx_b in gene_to_indices[gene_b]:
            source_nodes.extend([idx_a, idx_b])
            target_nodes.extend([idx_b, idx_a])

print("Injecting intra-gene connections...")
# Add intra-gene interactions (e.g., SNCA_TSS200 <-> SNCA_Body)
for gene, indices in gene_to_indices.items():
    if len(indices) > 1:
        for idx_a, idx_b in itertools.permutations(indices, 2):
            source_nodes.append(idx_a)
            target_nodes.append(idx_b)

edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
print(f"Edges before physical self-loops: {edge_index.shape[1]}")

# Add physical self-loops (Node A -> Node A)
edge_index, _ = add_self_loops(edge_index, num_nodes=len(ordered_regions))

print(f"Final edge_index shape: {edge_index.shape}")
torch.save(edge_index, "/workspace/results/global_string_edge_index.pt")

/opt/conda/envs/rapids-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading patient metadata...
Loading STRING links and filtering by high confidence...
Loading STRING aliases and matching to our genes...
Merging interactions...
Translating global edges to node indices...
Injecting intra-gene connections...
Edges before physical self-loops: 21505276
Final edge_index shape: torch.Size([2, 21595675])


In [5]:
import torch
import json
import numpy as np
import pandas as pd
from torch_geometric.data import Dataset, Data
from safetensors.torch import load_file

class MethylationGraphDataset(Dataset):
    def __init__(self, safetensors_path, meta_path, edge_index_path, pheno_path):
        super().__init__()

        # 1. Load the shared graph structure (Adjacency Matrix)
        self.edge_index = torch.load(edge_index_path, weights_only=True)

        # 2. Load the node features via memory-mapping (fast and RAM-efficient)
        self.patient_tensors = load_file(safetensors_path)["patient_graphs"]

        # 3. Load metadata to map the tensor indices back to exact Sample IDs
        with open(meta_path, "r") as f:
            self.sample_ids = json.load(f)["sample_ids"]

        # 4. Load phenotype data (Labels and Cell Proportions)
        # Using Pandas here because DataLoader workers operate on the CPU before GPU transfer
        self.pheno_df = pd.read_parquet(pheno_path)

        # Ensure the index matches the sample_ids format for instant lookups
        if self.pheno_df.index.name != "Sample_Name":
            self.pheno_df = self.pheno_df.set_index("Sample_Name")

        # Support either one-hot encoded labels (Sample_Group_*) or a single Sample_Group column.
        self.one_hot_group_cols = [c for c in self.pheno_df.columns if str(c).startswith("Sample_Group_")]

    def __len__(self):
        return len(self.sample_ids)

    def __getitem__(self, idx):
        # Identify the patient
        sample_id = self.sample_ids[idx]

        # Extract the [90399, 2] Node Feature matrix (X)
        x = self.patient_tensors[idx]

        # Extract tabular metadata for this patient
        pheno_row = self.pheno_df.loc[sample_id]

        # Define the target label (y).
        if self.one_hot_group_cols:
            label = int(pheno_row[self.one_hot_group_cols].to_numpy(dtype=float).argmax())
        else:
            group_val = pheno_row["Sample_Group"]
            if pd.isna(group_val):
                raise ValueError(f"Sample_Group is missing for sample '{sample_id}'")

            if isinstance(group_val, str):
                normalized = group_val.strip()
                label_map = {"Control": 0, "PD": 1}
                if normalized in label_map:
                    label = label_map[normalized]
                else:
                    try:
                        label = int(float(normalized))
                    except ValueError as e:
                        raise ValueError(
                            f"Unsupported Sample_Group value '{group_val}' for sample '{sample_id}'. "
                            "Use numeric labels, Control/PD strings, or one-hot Sample_Group_* columns."
                        ) from e
            else:
                label = int(group_val)

        y = torch.tensor([label], dtype=torch.long)

        # Extract the Houseman cell proportions
        cell_cols = ['CD8T', 'CD4T', 'NK', 'Bcell', 'Mono', 'Gran']
        cell_array = np.asarray(pheno_row[cell_cols].values, dtype=np.float32)
        cell_props = torch.from_numpy(cell_array).unsqueeze(0)

        # Construct the official PyTorch Geometric Data object
        # 'u' is PyG's standard attribute for graph-level global features
        data = Data(
            x=x,
            edge_index=self.edge_index,
            y=y,
            u=cell_props
        )

        return data

In [6]:
from torch_geometric.loader import DataLoader

# Initialize the dataset
peg1_dataset = MethylationGraphDataset(
    safetensors_path="/workspace/results/patient_graphs.safetensors",
    meta_path="/workspace/results/patient_graphs_meta.json",
    edge_index_path="/workspace/results/global_string_edge_index.pt",
    pheno_path="/workspace/data/GSE111629_pheno_data.parquet"
)

# Create the loader
# batch_size=1: each graph already has ~21.6M edges (STRING PPI expanded to CpG-level
# nodes), so conv1's per-edge gather (edges * hidden_dim floats) blows past 22GB VRAM
# at batch_size=8 (172.7M edges -> ~41GB). Raise this only after reducing edge_index size.
train_loader = DataLoader(peg1_dataset, batch_size=1, shuffle=True)

# Test the pipeline
first_batch = next(iter(train_loader))
print(first_batch)
# Expected Output similar to:
# DataBatch(x=[723192, 2], edge_index=[2, E_total], y=[8], u=[8, 6], batch=[723192], ptr=[9])

DataBatch(x=[90399, 2], edge_index=[2, 21595675], y=[1], u=[1, 6], batch=[90399], ptr=[2])


In [7]:
import torch
import torch.nn.functional as F
from torch.nn import Linear, Sequential, ReLU, Dropout, BatchNorm1d
from torch_geometric.nn import SAGEConv, global_mean_pool

class MethylationSAGE(torch.nn.Module):
    def __init__(self, node_in_dim=2, hidden_dim=64, cell_prop_dim=6, num_classes=2):
        super().__init__()
        
        # Project [mean, var] to hidden dimensions
        self.node_proj = Linear(node_in_dim, hidden_dim)
        
        # SAGEConv calculates neighborhood updates without the massive memory overhead of Attention
        self.conv1 = SAGEConv(hidden_dim, hidden_dim)
        self.bn1 = BatchNorm1d(hidden_dim)
        
        self.conv2 = SAGEConv(hidden_dim, hidden_dim)
        self.bn2 = BatchNorm1d(hidden_dim)
        
        # Late-Fusion Classifier
        fusion_dim = hidden_dim + cell_prop_dim
        self.classifier = Sequential(
            Linear(fusion_dim, 32),
            ReLU(),
            Dropout(p=0.5),
            Linear(32, num_classes)
        )

    def forward(self, x, edge_index, u, batch):
        
        x = self.node_proj(x)
        x = F.elu(x)
        
        # Message Passing
        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.elu(x)
        x = F.dropout(x, p=0.4, training=self.training)
        
        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.elu(x)
        
        # Pool the 90,399 nodes into a single patient vector
        x_pool = global_mean_pool(x, batch)
        
        # Fuse with the Houseman cell proportions
        if u.dim() == 3:
            u = u.squeeze(1) 
            
        fused_vector = torch.cat([x_pool, u], dim=1)
        logits = self.classifier(fused_vector)
        
        return logits

In [ ]:
%pip install torch torchvision torchaudio \
    --index-url https://download.pytorch.org/whl/cu124


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cpu":
    print("CUDA is unavailable in this Torch build; running inference on CPU.")

model = MethylationSAGE().to(device)
batch = first_batch.to(device)
predictions = model(batch.x, batch.edge_index, batch.u, batch.batch)
print(predictions.shape)

Using device: cuda
torch.Size([1, 2])


In [9]:
import torch
import sys
import subprocess

print("Python:", sys.version)
print("Torch:", torch.__version__)
print("Torch CUDA runtime tag:", torch.version.cuda)
print("torch.cuda.is_available():", torch.cuda.is_available())

try:
    out = subprocess.check_output(["nvidia-smi", "-L"], text=True)
    print("nvidia-smi detected GPUs:")
    print(out)
except Exception as e:
    print("nvidia-smi check failed:", e)

Python: 3.13.15 | packaged by conda-forge | (main, Aug 10 2026, 13:05:01) [GCC 14.4.0]
Torch: 2.6.0+cu124
Torch CUDA runtime tag: 12.4
torch.cuda.is_available(): True
nvidia-smi detected GPUs:
GPU 0: NVIDIA L4 (UUID: GPU-bc19965f-5567-95c4-1315-1c09869cc992)



In [ ]:
# Force CUDA-enabled PyTorch build (cu124) into the active notebook environment
%pip uninstall -y torch torchvision torchaudio
%pip install --no-cache-dir --force-reinstall torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu124
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 134.1 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 504.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 262.7 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 373.0 MB/s  0:00:0100:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 189.7 MB/s  0:00:02a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [10]:
import torch
import torch.nn as nn
from torch.optim import Adam

# 1. Setup the Device, Model, Optimizer, and Loss
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MethylationSAGE().to(device)
class_weights = torch.tensor([0.89, 1.13], dtype=torch.float32).to(device)
# CrossEntropyLoss is the standard for multi-class or binary classification outputting logits
criterion = nn.CrossEntropyLoss(weight=class_weights)

# Adam is the standard optimizer for deep learning. 
# lr = learning rate (how large of a step the optimizer takes when updating weights)
# weight_decay = L2 regularization to prevent the model from memorizing the training data
optimizer = Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)


# 2. Define the Training Loop Parameters
num_epochs = 20
print(f"Starting training on {device}...")

for epoch in range(num_epochs):
    # Set the model to training mode (enables Dropout and Batch Normalization)
    model.train()
    
    total_loss = 0.0
    correct_predictions = 0
    total_samples = 0
    
    for batch_idx, batch in enumerate(train_loader):
        # Move the entire batched graph and labels to the GPU
        batch = batch.to(device)
        
        # Step A: Clear old gradients from the previous step
        optimizer.zero_grad()
        
        # Step B: The Forward Pass
        # Returns shape: [batch_size, 2]
        logits = model(batch.x, batch.edge_index, batch.u, batch.batch)
        
        # Step C: Calculate the Error (Loss)
        # batch.y contains the true labels (0 for Control, 1 for PD)
        loss = criterion(logits, batch.y)
        
        # Step D: Backpropagation (Calculate gradients)
        loss.backward()
        
        # Step E: Update the model weights
        optimizer.step()
        
        # --- Tracking Metrics ---
        total_loss += loss.item() * batch.num_graphs
        total_samples += batch.num_graphs
        
        # Calculate accuracy for this batch
        # Logits are raw scores; the highest score is the model's chosen class
        predicted_classes = logits.argmax(dim=1)
        correct_predictions += int((predicted_classes == batch.y).sum())
        
    # Calculate epoch-level metrics
    epoch_loss = total_loss / total_samples
    epoch_acc = correct_predictions / total_samples
    
    print(f"Epoch {epoch+1:03d}/{num_epochs:03d} | Loss: {epoch_loss:.4f} | Accuracy: {epoch_acc:.4f}")

Starting training on cuda...
Epoch 001/020 | Loss: 0.6957 | Accuracy: 0.4795
Epoch 002/020 | Loss: 0.6840 | Accuracy: 0.5597
Epoch 003/020 | Loss: 0.6893 | Accuracy: 0.5504


KeyboardInterrupt: 